# Appendix - 데이터셋 상세 분석

이 노트북은 각 데이터셋에 대한 자세한 탐색적 분석을 포함합니다.
주요 노트북에서는 핵심 인사이트만 다루고, 여기서는 모든 변수에 대한 상세 분석을 진행합니다.

## 0. 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import requests
from io import StringIO

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

# 데이터 로드 (간단 버전)
BASE_URL = "https://raw.githubusercontent.com/tashydean/Basic_Health_Care/refs/heads/master/Data_Files/"

def quick_load(filename, schema):
    url = f"{BASE_URL}{filename}"
    response = requests.get(url)
    return pd.read_csv(StringIO(response.text), sep='\t', skiprows=1, 
                      header=None, names=schema, dtype=str)

df_patient = quick_load(
    "PatientCorePopulatedTable.txt",
    ["PatientID", "PatientGender", "PatientDateOfBirth", "PatientRace", 
     "PatientMaritalStatus", "PatientLanguage", "PatientPopulationPercentageBelowPoverty"]
)

df_admissions = quick_load(
    "AdmissionsCorePopulatedTable.txt",
    ["PatientID", "AdmissionID", "AdmissionStartDate", "AdmissionEndDate"]
)

df_diagnoses = quick_load(
    "AdmissionsDiagnosesCorePopulatedTable.txt",
    ["PatientID", "AdmissionID", "PrimaryDiagnosisCode", "PrimaryDiagnosisDescription"]
)

df_labs = quick_load(
    "LabsCorePopulatedTable.txt",
    ["PatientID", "AdmissionID", "LabName", "LabValue", "LabUnits", "LabDateTime"]
)

# 타입 변환
df_patient['PatientDateOfBirth'] = pd.to_datetime(df_patient['PatientDateOfBirth'])
df_patient['PatientPopulationPercentageBelowPoverty'] = pd.to_numeric(
    df_patient['PatientPopulationPercentageBelowPoverty']
)
df_admissions['AdmissionStartDate'] = pd.to_datetime(df_admissions['AdmissionStartDate'])
df_admissions['AdmissionEndDate'] = pd.to_datetime(df_admissions['AdmissionEndDate'])
df_labs['LabDateTime'] = pd.to_datetime(df_labs['LabDateTime'])
df_labs['LabValue'] = pd.to_numeric(df_labs['LabValue'])

print("✅ 데이터 로드 완료")

## 1. 환자 데이터 상세 분석

### 1.1 성별 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 성별 분포
gender_counts = df_patient['PatientGender'].value_counts()
axes[0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', startangle=90,
           colors=['lightblue', 'lightcoral'])
axes[0].set_title('성별 분포', fontsize=14, fontweight='bold')

# 막대 그래프
gender_counts.plot(kind='bar', ax=axes[1], color=['lightblue', 'lightcoral'], alpha=0.7)
axes[1].set_title('성별 환자 수', fontsize=14, fontweight='bold')
axes[1].set_xlabel('성별')
axes[1].set_ylabel('환자 수')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

for i, v in enumerate(gender_counts):
    axes[1].text(i, v + 1, str(v), ha='center')

plt.tight_layout()
plt.show()

print("\n📊 성별 통계:")
print(gender_counts)

### 1.2 인종 분포

In [ ]:
race_counts = df_patient['PatientRace'].value_counts()

plt.figure(figsize=(10, 6))
race_counts.plot(kind='barh', color='skyblue', alpha=0.7)
plt.title('인종별 환자 분포', fontsize=14, fontweight='bold')
plt.xlabel('환자 수')
plt.ylabel('인종')

for i, v in enumerate(race_counts):
    plt.text(v + 0.5, i, str(v), va='center')

plt.tight_layout()
plt.show()

print("\n📊 인종 통계:")
print(race_counts)

### 1.3 결혼 상태 분포

In [ ]:
marital_counts = df_patient['PatientMaritalStatus'].value_counts()

plt.figure(figsize=(10, 6))
marital_counts.plot(kind='bar', color='lightgreen', alpha=0.7)
plt.title('결혼 상태별 환자 분포', fontsize=14, fontweight='bold')
plt.xlabel('결혼 상태')
plt.ylabel('환자 수')
plt.xticks(rotation=45)

for i, v in enumerate(marital_counts):
    plt.text(i, v + 1, str(v), ha='center')

plt.tight_layout()
plt.show()

print("\n📊 결혼 상태 통계:")
print(marital_counts)

### 1.4 사용 언어 분포

In [ ]:
language_counts = df_patient['PatientLanguage'].value_counts()

plt.figure(figsize=(10, 6))
language_counts.plot(kind='bar', color='salmon', alpha=0.7)
plt.title('사용 언어별 환자 분포', fontsize=14, fontweight='bold')
plt.xlabel('언어')
plt.ylabel('환자 수')
plt.xticks(rotation=45)

for i, v in enumerate(language_counts):
    plt.text(i, v + 1, str(v), ha='center')

plt.tight_layout()
plt.show()

print("\n📊 언어 통계:")
print(language_counts)

### 1.5 빈곤율 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 히스토그램
df_patient['PatientPopulationPercentageBelowPoverty'].hist(bins=20, ax=axes[0], 
                                                            color='mediumpurple', alpha=0.7)
axes[0].set_title('빈곤율 분포', fontsize=14, fontweight='bold')
axes[0].set_xlabel('빈곤율 (%)')
axes[0].set_ylabel('환자 수')
axes[0].axvline(df_patient['PatientPopulationPercentageBelowPoverty'].mean(), 
               color='red', linestyle='--', label='평균')
axes[0].axvline(df_patient['PatientPopulationPercentageBelowPoverty'].median(), 
               color='green', linestyle='--', label='중앙값')
axes[0].legend()

# 박스플롯
df_patient.boxplot(column='PatientPopulationPercentageBelowPoverty', ax=axes[1])
axes[1].set_title('빈곤율 박스플롯', fontsize=14, fontweight='bold')
axes[1].set_ylabel('빈곤율 (%)')

plt.tight_layout()
plt.show()

print("\n📊 빈곤율 통계:")
print(df_patient['PatientPopulationPercentageBelowPoverty'].describe())

### 1.6 연령 분포

In [ ]:
# 첫 입원일 기준 나이 계산
first_admission = df_admissions.groupby('PatientID')['AdmissionStartDate'].min()
df_patient_age = df_patient.copy()
df_patient_age['FirstAdmissionDate'] = df_patient_age['PatientID'].map(first_admission)
df_patient_age['Age'] = (
    df_patient_age['FirstAdmissionDate'] - df_patient_age['PatientDateOfBirth']
).dt.total_seconds() / (365.25 * 24 * 3600)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 히스토그램
df_patient_age['Age'].hist(bins=20, ax=axes[0], color='teal', alpha=0.7)
axes[0].set_title('연령 분포', fontsize=14, fontweight='bold')
axes[0].set_xlabel('나이 (세)')
axes[0].set_ylabel('환자 수')
axes[0].axvline(df_patient_age['Age'].mean(), color='red', linestyle='--', label='평균')
axes[0].legend()

# 연령대별 분포
age_groups = pd.cut(df_patient_age['Age'], bins=[0, 30, 50, 70, 100], 
                    labels=['0-30', '31-50', '51-70', '71+'])
age_group_counts = age_groups.value_counts().sort_index()
age_group_counts.plot(kind='bar', ax=axes[1], color='coral', alpha=0.7)
axes[1].set_title('연령대별 환자 수', fontsize=14, fontweight='bold')
axes[1].set_xlabel('연령대')
axes[1].set_ylabel('환자 수')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

for i, v in enumerate(age_group_counts):
    axes[1].text(i, v + 1, str(v), ha='center')

plt.tight_layout()
plt.show()

print("\n📊 연령 통계:")
print(df_patient_age['Age'].describe())

## 2. 입원 데이터 상세 분석

### 2.1 재원 기간 분포

In [ ]:
# 재원 기간 계산
df_admissions['LengthOfStay'] = (
    df_admissions['AdmissionEndDate'] - df_admissions['AdmissionStartDate']
).dt.total_seconds() / (24 * 3600)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 히스토그램
df_admissions['LengthOfStay'].hist(bins=30, ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('재원 기간 분포', fontsize=14, fontweight='bold')
axes[0].set_xlabel('재원 기간 (일)')
axes[0].set_ylabel('입원 건수')
axes[0].axvline(df_admissions['LengthOfStay'].mean(), color='red', linestyle='--', label='평균')
axes[0].legend()

# 박스플롯
df_admissions.boxplot(column='LengthOfStay', ax=axes[1])
axes[1].set_title('재원 기간 박스플롯', fontsize=14, fontweight='bold')
axes[1].set_ylabel('재원 기간 (일)')

plt.tight_layout()
plt.show()

print("\n📊 재원 기간 통계:")
print(df_admissions['LengthOfStay'].describe())

### 2.2 연도별 입원 추이

In [ ]:
df_admissions['Year'] = df_admissions['AdmissionStartDate'].dt.year
yearly = df_admissions['Year'].value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(yearly.index, yearly.values, marker='o', linewidth=2, markersize=8)
plt.title('연도별 입원 건수 추이', fontsize=14, fontweight='bold')
plt.xlabel('연도')
plt.ylabel('입원 건수')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 연도별 입원 건수:")
print(yearly)

### 2.3 계절별 입원 패턴

In [ ]:
df_admissions['Month'] = df_admissions['AdmissionStartDate'].dt.month
df_admissions['Season'] = pd.cut(
    df_admissions['Month'],
    bins=[0, 3, 6, 9, 12],
    labels=['겨울(1-3)', '봄(4-6)', '여름(7-9)', '가을(10-12)']
)

season_counts = df_admissions['Season'].value_counts()

plt.figure(figsize=(10, 6))
season_counts.plot(kind='bar', color=['lightblue', 'lightgreen', 'lightyellow', 'orange'], alpha=0.7)
plt.title('계절별 입원 건수', fontsize=14, fontweight='bold')
plt.xlabel('계절')
plt.ylabel('입원 건수')
plt.xticks(rotation=45)

for i, v in enumerate(season_counts):
    plt.text(i, v + 2, str(v), ha='center')

plt.tight_layout()
plt.show()

print("\n📊 계절별 입원 통계:")
print(season_counts)

## 3. 진단 데이터 상세 분석

### 3.1 상위 20개 진단

In [ ]:
top_diagnoses = df_diagnoses['PrimaryDiagnosisDescription'].value_counts().head(20)

plt.figure(figsize=(12, 8))
top_diagnoses.plot(kind='barh', color='mediumseagreen', alpha=0.7)
plt.title('상위 20개 진단', fontsize=14, fontweight='bold')
plt.xlabel('진단 건수')
plt.ylabel('진단명')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\n🔝 상위 20개 진단:")
print(top_diagnoses)

### 3.2 ICD-10 대분류별 진단 수

In [ ]:
df_diagnoses['Category'] = df_diagnoses['PrimaryDiagnosisCode'].str[0]
category_counts = df_diagnoses['Category'].value_counts()

plt.figure(figsize=(12, 6))
category_counts.plot(kind='bar', color='mediumpurple', alpha=0.7)
plt.title('ICD-10 대분류별 진단 건수', fontsize=14, fontweight='bold')
plt.xlabel('ICD-10 대분류 (첫 글자)')
plt.ylabel('진단 건수')
plt.xticks(rotation=0)

for i, v in enumerate(category_counts):
    plt.text(i, v + 2, str(v), ha='center')

plt.tight_layout()
plt.show()

print("\n📊 ICD-10 대분류 통계:")
print(category_counts)

## 4. 검사 데이터 상세 분석

### 4.1 검사 항목별 건수 및 평균값

In [ ]:
# 검사 항목별 통계
lab_stats = df_labs.groupby('LabName')['LabValue'].agg(['count', 'mean', 'std', 'min', 'max'])
lab_stats = lab_stats.sort_values('count', ascending=False)

print("📊 검사 항목별 통계:")
display(lab_stats.head(15))

# 검사 건수 시각화
plt.figure(figsize=(12, 8))
lab_stats['count'].head(15).plot(kind='barh', color='teal', alpha=0.7)
plt.title('상위 15개 검사 항목별 검사 건수', fontsize=14, fontweight='bold')
plt.xlabel('검사 건수')
plt.ylabel('검사 항목')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 4.2 주요 검사 항목별 분포

In [ ]:
# 상위 6개 검사 항목의 분포
top_labs = lab_stats.head(6).index

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, lab in enumerate(top_labs):
    lab_data = df_labs[df_labs['LabName'] == lab]['LabValue']
    axes[i].hist(lab_data, bins=30, color='skyblue', alpha=0.7, edgecolor='black')
    axes[i].set_title(lab, fontweight='bold')
    axes[i].set_xlabel('검사 값')
    axes[i].set_ylabel('빈도')
    axes[i].axvline(lab_data.mean(), color='red', linestyle='--', label='평균')
    axes[i].legend()

plt.suptitle('주요 검사 항목별 값 분포', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

### 4.3 검사 단위별 분류

In [ ]:
unit_counts = df_labs['LabUnits'].value_counts()

plt.figure(figsize=(10, 6))
unit_counts.plot(kind='bar', color='coral', alpha=0.7)
plt.title('검사 단위별 검사 건수', fontsize=14, fontweight='bold')
plt.xlabel('단위')
plt.ylabel('검사 건수')
plt.xticks(rotation=45)

for i, v in enumerate(unit_counts):
    plt.text(i, v + 500, str(v), ha='center')

plt.tight_layout()
plt.show()

print("\n📊 검사 단위 통계:")
print(unit_counts)

## 5. 교차 분석

### 5.1 성별 × 인종

In [ ]:
gender_race = pd.crosstab(df_patient['PatientGender'], df_patient['PatientRace'])

plt.figure(figsize=(10, 6))
gender_race.plot(kind='bar', stacked=True, alpha=0.8)
plt.title('성별 × 인종 교차 분포', fontsize=14, fontweight='bold')
plt.xlabel('성별')
plt.ylabel('환자 수')
plt.xticks(rotation=0)
plt.legend(title='인종')
plt.tight_layout()
plt.show()

print("\n📊 성별 × 인종 교차표:")
display(gender_race)

### 5.2 연령대 × 성별

In [ ]:
age_groups = pd.cut(df_patient_age['Age'], bins=[0, 30, 50, 70, 100], 
                    labels=['0-30', '31-50', '51-70', '71+'])
df_patient_age['AgeGroup'] = age_groups

age_gender = pd.crosstab(df_patient_age['AgeGroup'], df_patient['PatientGender'])

plt.figure(figsize=(10, 6))
age_gender.plot(kind='bar', color=['lightblue', 'lightcoral'], alpha=0.7)
plt.title('연령대 × 성별 분포', fontsize=14, fontweight='bold')
plt.xlabel('연령대')
plt.ylabel('환자 수')
plt.xticks(rotation=0)
plt.legend(title='성별')
plt.tight_layout()
plt.show()

print("\n📊 연령대 × 성별 교차표:")
display(age_gender)

## 6. 요약

이 Appendix에서는 각 데이터셋의 모든 변수에 대한 상세 분석을 수행했습니다.

### 주요 발견사항:
- 환자: 성별, 인종, 연령, 빈곤율의 다양한 분포
- 입원: 재원 기간, 계절적 패턴, 연도별 추이
- 진단: 다양한 질병 코드와 ICD-10 대분류
- 검사: 35가지 검사 항목, 각각의 정상 범위와 분포

이러한 상세 분석을 통해 데이터의 특성을 완전히 이해하고,
주요 노트북에서는 핵심 인사이트에 집중할 수 있습니다.